In [36]:
import os
import sys 
print("Active Environment Path:", sys.executable)
import json
import subprocess
import tempfile
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, PrecisionRecallDisplay

# local
os.chdir("/home/azureuser/EBRD_Tillage/AK_Experiments/AK_pipeline")
sys.path.append(os.getcwd())
from Generic.addSentinelIndices import calculateSentinelIndex

Active Environment Path: /anaconda/envs/geospatial-base/bin/python


In [ ]:

RUN_ACQUIRE_PATH = Path("/home/azureuser/EBRD_Tillage/EBRD_tillage_monitoring-main/src/acquire/run_acquire.py")  
TEMP_DIR = Path("/home/azureuser/EBRD_Tillage/data/temp")
TEMP_DIR.mkdir(parents=True, exist_ok=True)
INPUT_DATA_PATH= Path("/home/azureuser/EBRD_Tillage/data/input/EBRD_merged_classified.shp")  

# Global Variables 
kind = 'Sentinel2' # Kinda useless now, only Sentinel 2 in VM
band = 'STI'
padding = 5
tempAg = 'min'
spatAg = 'mean'
classCol = 'Classifica'
stratified_limitation = 10


In [23]:
# ==============================================================================
# 1. SETUP & PATH CONFIGURATION
# ==============================================================================
print(f"Reading shapefile from: {INPUT_DATA_PATH}")
gdf = gpd.read_file(INPUT_DATA_PATH)

# Quick check
print(f"Successfully loaded {len(gdf)} rows.")
gdf = gdf.rename(columns={'Beginning':'start_date',"End date":'end_date'})
print(gdf.columns)
gdf["start_date"] = pd.to_datetime(gdf["start_date"])
gdf["end_date"] = pd.to_datetime(gdf["end_date"])
gdf = gdf[(gdf.Classifica == 'Conservation') | (gdf.Classifica == 'Conventional')]

if(stratified_limitation >0): 
    good = gdf[gdf.Classifica == 'Conservation' ].sample(stratified_limitation)
    bad = gdf[gdf.Classifica == 'Conventional'].sample(stratified_limitation)
    gdf = pd.concat([good,bad],ignore_index=True).reset_index() 
    
print(f"Filtered to {len(gdf)} rows.")


Reading shapefile from: /home/azureuser/EBRD_Tillage/data/input/EBRD_merged_classified.shp
Successfully loaded 5746 rows.
Index(['index', 'fid', 'Operation', 'Group of o', 'Machinery', 'Machine',
       'start_date', 'end_date', 'Field ID', 'Crop', 'Year', 'Area, ha',
       'operation_', 'source_fil', 'FieldAreaH', 'Completion', 'Classifica',
       'geometry'],
      dtype='str')
Filtered to 20 rows.


In [41]:
# ==============================================================================
# 2. ITERATE OVER TILLAGE EVENTS & EXECUTE ACQUISITION PIPELINE
# ==============================================================================
event_band_results = []

for idx, row in gdf.iterrows():
    event_id = str(row["fid"])
    till_type = row["Classifica"]
    
    # Apply 10-day buffer to event dates
    buf_start = (row["start_date"] - pd.Timedelta(days=padding)).strftime("%Y-%m-%d")
    buf_end = (row["end_date"] + pd.Timedelta(days=padding)).strftime("%Y-%m-%d")
    
    # Save isolated event geometry to temp file
    event_gdf = gpd.GeoDataFrame([row], crs=gdf.crs)
    event_vector_path = TEMP_DIR / f"{event_id}.geojson"
    event_gdf.to_file(event_vector_path, driver="GeoJSON")
    
    # Build dynamic JSON config
    event_output_dir = TEMP_DIR / f"output_{event_id}"
    config_dict = {
        "fields_path": str(event_vector_path),
        "output_dir": str(event_output_dir),
        "field_id_col": "fid",
        "start_date": buf_start,
        "end_date": buf_end,
        "max_cloud_cover": 90,
        "resolution": 10,
        "assets": ["B02", "B03", "B04", "B05", "B06", "B07", "B08", "B8A", "B11", "B12", "SCL"],
        "max_workers": 1,
        "skip_existing": True
        #"scene_samples_subdir": "scene_samples",
        #"scene_spatial_batching": true,
        #"max_scene_window_pixels": 16000000,
    }
    
    
    config_path = TEMP_DIR / f"config_{event_id}.json"
    with open(config_path, "w", encoding="utf-8") as f:
        json.dump(config_dict, f, indent=2)
        
    print(f"\n--- Processing Event {idx+1}/{len(gdf)}: {event_id} ({till_type}) | Window: {buf_start} to {buf_end} ---")
    
    # Call run_acquire.py via subprocess CLI
    cmd = [
        sys.executable, str(RUN_ACQUIRE_PATH),
        "--source", "s2",
        "--config", str(config_path),
        "--steps", "inventory", "download"
    ]
    
    try:
        subprocess.run(cmd, check=True)
    except subprocess.CalledProcessError as e:
        print(f"Acquisition failed for event {event_id}: {e}")
        event_ndvi_results.append(np.nan)
        continue
    
    # ==========================================================================
    # 3. CALCULATE NDVI & TWO-STAGE AGGREGATION (TEMPORAL FIRST, THEN SPATIAL)
    # ==========================================================================
    parquet_files = list((event_output_dir / "scene_samples").glob("**/*.parquet"))
    
    if not parquet_files:
        print(f"No scene samples returned for event {event_id}.")
        event_ndvi_results.append(np.nan)
        continue

    # Load and combine extracted pixel data
    df_pixels = pd.concat([pd.read_parquet(f) for f in parquet_files], ignore_index=True)
    
    # Filter valid non-cloudy pixels
    df_valid = df_pixels[df_pixels["valid_px"] == True].copy()
    if df_valid.empty:
        print(f"Warning: All pixels invalid/cloudy. Falling back to unmasked pixels for {event_id}.")
        df_valid = df_pixels.copy()
        
        
    df_valid = calculateSentinelIndex(df_valid,band)
    
    # Stage 1: Temporal Aggregation First (min NDVI over time window for each pixel location)
    if tempAg == 'min':
        pixel_min_band = df_valid.groupby(["pixel_col", "pixel_row"])[band].min()
    else: 
        raise NotImplementedError 

    # Stage 2: Spatial Aggregation (spatial mean across field polygon)
    if spatAg == 'mean':
        event_summary_band = pixel_min_band.mean()
    else: 
        raise NotImplementedError
    
    event_band_results.append(event_summary_band)
    print(f"Resulting Event {band} (Temporal Min -> Spatial Mean): {event_summary_band:.4f}")

# Attach summary NDVI values back to primary GeoDataFrame
gdf["summary_band"] = event_ndvi_results

# Filter out failed/missing rows for threshold analysis
analysis_df = gdf.dropna(subset=["summary_ndvi"]).copy()
analysis_df




--- Processing Event 1/20: 6968.0 (Conservation) | Window: 2023-08-05 to 2023-08-16 ---
Running: /anaconda/envs/geospatial-base/bin/python download_s2_pc.py --config /tmp/tmp8loq7hsy.json --stac-inventory-only
[     0.0s] START: Searching full STAC inventory | RAM=546.2 MB
[     0.2s] DONE:  Searching full STAC inventory (0.1s) | RAM=551.8 MB | dRAM=+5.6 MB
[     0.2s] Found 9 Sentinel-2 items in full inventory search
[     0.2s] START: Writing STAC item inventory: stac_inventory_items.csv | RAM=551.8 MB
[     0.2s] DONE:  Writing STAC item inventory: stac_inventory_items.csv (0.0s) | RAM=552.5 MB | dRAM=+0.7 MB
[     0.2s] START: Writing STAC item footprints: stac_inventory_items.geojson | RAM=552.5 MB
[     0.2s] DONE:  Writing STAC item footprints: stac_inventory_items.geojson (0.0s) | RAM=553.3 MB | dRAM=+0.8 MB
[     0.2s] START: Writing full STAC item cache: stac_inventory_items.jsonl | RAM=553.3 MB
[     0.2s] DONE:  Writing full STAC item cache: stac_inventory_items.jsonl (0.0

Traceback (most recent call last):
  File "/home/azureuser/EBRD_Tillage/EBRD_tillage_monitoring-main/src/acquire/download_s2_pc.py", line 947, in <module>
    main()
  File "/home/azureuser/EBRD_Tillage/EBRD_tillage_monitoring-main/src/acquire/download_s2_pc.py", line 934, in main
    run_scene_sampling(
  File "/home/azureuser/EBRD_Tillage/EBRD_tillage_monitoring-main/src/acquire/download_s2_pc.py", line 712, in run_scene_sampling
    scene_jobs = load_cached_scene_jobs(output_dir)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/azureuser/EBRD_Tillage/EBRD_tillage_monitoring-main/src/acquire/download_s2_pc.py", line 400, in load_cached_scene_jobs
    raise FileNotFoundError(
FileNotFoundError: Missing STAC cache files. Run with --stac-inventory-only first. Missing: ['/home/azureuser/EBRD_Tillage/data/temp/output_8469.0/stac_inventory_items.jsonl', '/home/azureuser/EBRD_Tillage/data/temp/output_8469.0/stac_field_item_map.csv']
Traceback (most recent call last):
  File

Acquisition failed for event 8469.0: Command '['/anaconda/envs/geospatial-base/bin/python', '/home/azureuser/EBRD_Tillage/EBRD_tillage_monitoring-main/src/acquire/run_acquire.py', '--source', 's2', '--config', '/home/azureuser/EBRD_Tillage/data/temp/config_8469.0.json', '--steps', 'inventory', 'download']' returned non-zero exit status 1.

--- Processing Event 20/20: 726.0 (Conventional) | Window: 2025-04-13 to 2025-04-24 ---
Running: /anaconda/envs/geospatial-base/bin/python download_s2_pc.py --config /tmp/tmp2_utmmdt.json --stac-inventory-only
[     0.0s] START: Searching full STAC inventory | RAM=546.1 MB
[     0.2s] DONE:  Searching full STAC inventory (0.1s) | RAM=551.4 MB | dRAM=+5.3 MB
[     0.2s] Found 7 Sentinel-2 items in full inventory search
[     0.2s] START: Writing STAC item inventory: stac_inventory_items.csv | RAM=551.4 MB
[     0.2s] DONE:  Writing STAC item inventory: stac_inventory_items.csv (0.0s) | RAM=552.2 MB | dRAM=+0.7 MB
[     0.2s] START: Writing STAC item f

ValueError: Length of values (2) does not match length of index (20)

IndentationError: unexpected indent (3075945151.py, line 4)